In [ ]:
import datasets
from datasets import load_dataset

huggingface_mrpc_dataset = load_dataset('glue', 'mrpc')
print(huggingface_mrpc_dataset)

In [ ]:
train = huggingface_mrpc_dataset['train']
cols = train.column_names
cols

In [ ]:
for i in range(5):
    for col in cols:
        print(col, ":", train[col][i])
    print('\n')

In [ ]:
import pandas as pd
from datasets import Dataset

def parse_mrpc_file(file_path):
    """MRPC 파일을 안전하게 파싱하는 함수"""
    data = {
        'Quality': [],
        '#1 ID': [],
        '#2 ID': [], 
        '#1 String': [],
        '#2 String': []
    }
    
    with open(file_path, 'r', encoding='utf-8') as f:
        # 헤더 스킵
        next(f)
        
        for line_num, line in enumerate(f, 1):
            try:
                parts = line.strip().split('\t')
                if len(parts) >= 5:
                    # 5개 컬럼으로 분할 (마지막 탭들은 모두 마지막 컬럼에 포함)
                    quality = int(parts[0])
                    id1 = int(parts[1]) 
                    id2 = int(parts[2])
                    string1 = parts[3]
                    string2 = '\t'.join(parts[4:])  # 나머지 모든 부분을 합침
                    
                    data['Quality'].append(quality)
                    data['#1 ID'].append(id1)
                    data['#2 ID'].append(id2)
                    data['#1 String'].append(string1)
                    data['#2 String'].append(string2)
                else:
                    print(f"Line {line_num}: Invalid format, skipping")
            except Exception as e:
                print(f"Line {line_num}: Error {e}, skipping")
    
    return pd.DataFrame(data)

# 로컬 파일에서 MRPC 데이터 읽기
train_df = parse_mrpc_file('data/msr_paraphrase_train.txt')
test_df = parse_mrpc_file('data/msr_paraphrase_test.txt')

In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# DataFrame을 dict 형식으로 변경 (각 컬럼을 리스트로 변환)
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')

# train 데이터를 train(80%)과 validation(20%)으로 분할
train_indices = list(range(len(train_dataset['Quality'])))
train_idx, val_idx = train_test_split(train_indices, test_size=0.2, random_state=42, 
                                      stratify=train_dataset['Quality'])

# validation 데이터셋 생성
validation_dataset = {}
for key in train_dataset.keys():
    validation_dataset[key] = [train_dataset[key][i] for i in val_idx]

# train 데이터셋 업데이트 (validation으로 사용된 데이터 제거)
train_dataset_final = {}
for key in train_dataset.keys():
    train_dataset_final[key] = [train_dataset[key][i] for i in train_idx]

# 허깅페이스 Dataset 객체로 변환
train_hf_dataset = Dataset.from_dict(train_dataset_final)
validation_hf_dataset = Dataset.from_dict(validation_dataset)
test_hf_dataset = Dataset.from_dict(test_dataset)

# DatasetDict 생성 (이게 허깅페이스의 표준 방식)
customized_mrpc_dataset = DatasetDict({
    'train': train_hf_dataset,
    'validation': validation_hf_dataset,
    'test': test_hf_dataset
})

# 결과 출력
print("DatasetDict({")
for split_name, split_data in customized_mrpc_dataset.items():
    print(f"    {split_name}: Dataset({{")
    print(f"        features: {list(split_data.features.keys())},")
    print(f"        num_rows: {split_data.num_rows}")
    print("    })")
print("})")

print("\n데이터셋 정보 확인!")
print(f"Train dataset shape: ({customized_mrpc_dataset['train'].num_rows}, {len(customized_mrpc_dataset['train'].features)})")
print(f"Validation dataset shape: ({customized_mrpc_dataset['validation'].num_rows}, {len(customized_mrpc_dataset['validation'].features)})")
print(f"Test dataset shape: ({customized_mrpc_dataset['test'].num_rows}, {len(customized_mrpc_dataset['test'].features)})")

print("Dataset 생성 완료!")
print(f"Train samples: {len(customized_mrpc_dataset['train'])}")
print(f"Validation samples: {len(customized_mrpc_dataset['validation'])}")
print(f"Test samples: {len(customized_mrpc_dataset['test'])}")

# 첫 번째 샘플 확인
print(f"\n첫 번째 train 샘플:")
print(customized_mrpc_dataset['train'][0])

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

huggingface_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
huggingface_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 2)

In [ ]:
def transform(data):
    return huggingface_tokenizer(
        data['sentence1'],
        data['sentence2'],
        truncation = True,
        padding = 'max_length',
        return_token_type_ids = False,
        )

In [ ]:
hf_dataset = huggingface_mrpc_dataset.map(transform, batched=True)

# train & validation & test split
hf_train_dataset = hf_dataset['train']
hf_val_dataset = hf_dataset['validation']
hf_test_dataset = hf_dataset['test']

In [ ]:
# Q. tf_train_dataset에 transform 함수를 매핑해봅시다. 어떤 오류가 발생하나요?
tf_train_dataset_error = tf_train_dataset.map(transform)

In [ ]:
# DataFrame을 dict 형식으로 변경 (각 컬럼을 리스트로 변환)
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')

# validation set이 없는 경우, train set에서 일부를 분할
# 또는 test set을 validation으로 사용
val_dataset = test_dataset.copy()  # 예시로 test를 val로 복사

# Huggingface Dataset 생성
train_dataset = Dataset.from_dict(train_dataset)
val_dataset = Dataset.from_dict(val_dataset) 
test_dataset = Dataset.from_dict(test_dataset)

print("Dataset 생성 완료!")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# 커스텀 데이터용 transform 함수 정의
def transform_custom(batch):
    # 커스텀 파일 데이터는 이미 문자열이므로 decode 불필요
    sentence1 = batch['#1 String']
    sentence2 = batch['#2 String']
    return huggingface_tokenizer(
        sentence1,
        sentence2,
        truncation=True,
        padding='max_length',
        return_token_type_ids=False,
    )

# 토큰화 및 패딩을 적용
train_dataset = train_dataset.map(transform_custom, batched=True)
val_dataset = val_dataset.map(transform_custom, batched=True)
test_dataset = test_dataset.map(transform_custom, batched=True)

In [ ]:
import os
import numpy as np
from transformers import Trainer, TrainingArguments

output_dir = 'transformers'

training_arguments = TrainingArguments(
    output_dir,                                         # output이 저장될 경로
    eval_strategy="epoch",           #evaluation하는 빈도
    learning_rate = 2e-5,                         #learning_rate
    per_device_train_batch_size = 8,   # 각 device 당 batch size
    per_device_eval_batch_size = 8,    # evaluation 시에 batch size
    num_train_epochs = 3,                     # train 시킬 총 epochs
    weight_decay = 0.01,                        # weight decay
)

In [ ]:
from evaluate import load
metric = load('glue', 'mrpc')

def compute_metrics(eval_pred):
    predictions,labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references = labels)

In [ ]:
trainer = Trainer(
    model=huggingface_model,           # 학습시킬 model
    args=training_arguments,           # TrainingArguments을 통해 설정한 arguments
    train_dataset=hf_train_dataset,    # training dataset
    eval_dataset=hf_val_dataset,       # evaluation dataset
    compute_metrics=compute_metrics,
)
trainer.train()
print("슝~")

In [ ]:
trainer.evaluate(hf_test_dataset)

In [ ]:
#메모리를 비워줍니다.
del huggingface_model

In [ ]:
# Q. 커스텀 데이터셋으로 학습시켜봅시다.
huggingface_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2)

trainer_custom = Trainer(
    model=huggingface_model,
    args=training_arguments,
    train_dataset=tf_train_dataset,
    eval_dataset=tf_val_dataset,
    compute_metrics=compute_metrics,
)
trainer_custom.train()

In [ ]:
# Validation 데이터셋을 이용해 평가해봅니다.
eval_results = trainer_custom.evaluate()

print(eval_results)